In [3]:
import re
import json
import sqlite3
import requests
from datetime import datetime
from bs4 import BeautifulSoup
from urllib.parse import urlparse

In [4]:
# optional, only needed for JS-heavy sites
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    import time
    HAS_SELENIUM = True
except ImportError:
    HAS_SELENIUM = False

import ollama

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
DB_PATH = r"D:\Github\Job_Finding_AI_Agent\job_search.db"
OLLAMA_MODEL = "llama3.2"  # change to your model


In [9]:
def fetch_page(url, use_selenium = False):
    """Fetch page text. Try BS4 first, Selenium if needed."""

    if use_selenium:
        if not HAS_SELENIUM:
            raise RuntimeError("Selenium not installed. pip install selenium")
        options = Options() # Selenium config object
        options.add_argument("--headless")
        options.add_argument("--no-sandbox")
        driver = webdriver.Chrome(options=options) # command to launch Chrome
        try:
            driver.get(url)
            time.sleep(4)  # wait for JS
            text = driver.find_element(By.TAG_NAME, "body").text
            return text[:10000]
        finally:
            driver.quit()

    # BS4
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
        )
    }
    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text[:10000]

## Test
# bad_url = "https://careers.intuitive.com/en/jobs/744000095572186/JOB210150/ai-research-intern/?source=linkedin&src=LinkedIn&trid=undefined"
good_test = "https://www.metacareers.com/profile/job_details/2082160555965742"
print(fetch_page(good_test, use_selenium=True))

Back to Jobs
Research Scientist Intern, Multimodal AI
Redmond, WA
Artificial Intelligence
+4 more
Apply now
The Meta Reality Labs Research Team brings together a world-class team of researchers, developers, and engineers to create the future of virtual and augmented reality, which together will become as universal and essential as smartphones and personal computers are today. And just as personal computers have done over the past 45 years, AR, VR and MR will ultimately change everything about how we work, play, and connect. We are developing all the technologies needed to enable breakthrough AR glasses and VR headsets, including optics and displays, computer vision, audio, graphics, brain-computer interfaces, haptic interaction, eye/hand/face/body tracking, perception science, and true telepresence. Some of those will advance much faster than others, but they all need to happen to enable AR, VR and MR that are so compelling that they become an integral part of our lives.

In particular